# Backtest 01 — Sell +25Δ OTM Vol, All Hours

**Strategy:** At the start of each UTC hour, sell 1,000 Kalshi BTC binary call contracts at the strike closest to +25Δ OTM (digi_px ≈ 25 cents). Delta-hedge with BTC every 5 minutes. Hold through expiration.

**Costs:** 5 bps on contract entry · 1 bps per BTC rebalance and final unwind

**Position size:** 1% × $100 000 starting portfolio = $1 000 notional (1 000 contracts × $1 face value)

**Binary call delta formula:** `Δ = n(d₂) / (S·σ·√T)` — BTC units to hold long = N × Δ

---

In [38]:
from __future__ import annotations
import functools
import warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from scipy.stats import norm
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:.4f}".format


In [39]:
DATA_DIR            = Path("../data")
STARTING_PORTFOLIO  = 100_000.0   # $100 000 starting cash
N_CONTRACTS           = 1_000   # 1% × $100k / $1 face value = 1 000 contracts
KALSHI_TAKER_RATE     = 0.07    # Kalshi taker fee multiplier: fee = 0.07 × C × P × (1−P)
                                 # peaks at P=0.50 → 1.75¢/contract; near-zero deep OTM/ITM
CONTRACT_SLIPPAGE_BPS = 10      # 10 bps market impact on premium (bid-ask cross)
BTC_SLIPPAGE_BPS      = 1       # 1 bps per BTC rebalance trade (and unwind)
HEDGE_INTERVAL_MINS = 5           # delta rebalance every 5 minutes
TARGET_DELTA_CENTS  = 25          # sell the +25Δ OTM contract (digi_px ≈ 25)
HOURS_IN_YEAR       = 8_760.0
MINS_PER_YEAR       = 525_960.0   # 365.25 × 24 × 60 — used to annualise 1m realised vol
IV_MIN              = 0.20        # 20% annualised floor — matches silver ETL;
                                  # below this the binary delta blows up (σ→0)
# ── Improvement filters ──────────────────────────────────────────────
IV_RVOL_RATIO_MIN   = 1.20   # only sell when implied_vol / realised_vol_1h ≥ this;
                              # ensures we are compensated for gamma risk
MIN_PREMIUM_CENTS   = 10     # skip contracts priced below 10 cents — TC dominates
EARLY_EXIT_THRESHOLD = 0.50  # take profit when contract price ≤ 50% of entry price;
                              # avoids gamma blowup in the final minutes
RV_LOOKBACK_MINS    = 60     # realised-vol look-back window (minutes)


In [40]:
@functools.lru_cache(maxsize=1)
def load_vol_surface() -> pl.DataFrame:
    df = pl.read_parquet(DATA_DIR / "silver/vol_surface.parquet")
    return df.with_columns([
        pl.col("snapshot").cast(pl.Utf8),
        pl.col("digi_contract_name").cast(pl.Utf8),
    ])

@functools.lru_cache(maxsize=1)
def load_binance() -> pl.DataFrame:
    return (
        pl.read_parquet(DATA_DIR / "bronze/binance_btc_1m.parquet")
        .with_columns(pl.col("close").cast(pl.Float64))
        .sort("timestamp")
    )

surface = load_vol_surface()
binance = load_binance()

print(f"Vol surface : {surface.shape[0]:>10,} rows | "
      f"{surface['expiry_time'].n_unique()} unique expiries")
print(f"Binance 1m  : {binance.shape[0]:>10,} rows | "
      f"{binance['timestamp'].min()} → {binance['timestamp'].max()}")
print(f"Snapshots   : {sorted(surface['snapshot'].unique().to_list())}")


Vol surface :    123,150 rows | 1380 unique expiries
Binance 1m  :    375,000 rows | 2025-08-31 22:00:00+00:00 → 2026-05-19 07:59:00+00:00
Snapshots   : ['T+1', 'T+10', 'T+11', 'T+12', 'T+2', 'T+3', 'T+4', 'T+5', 'T+6', 'T+7', 'T+8', 'T+9', 'T-1', 'T-10', 'T-11', 'T-2', 'T-3', 'T-4', 'T-5', 'T-6', 'T-7', 'T-8', 'T-9', 'T0']


In [41]:
# ── Utilities ────────────────────────────────────────────────────────

def _to_utc(dt: datetime) -> datetime:
    """Ensure datetime has UTC timezone."""
    return dt if dt.tzinfo is not None else dt.replace(tzinfo=timezone.utc)


def binary_call_delta(S: float, K: float, sigma: float, T: float) -> float:
    """
    Hedge delta of a binary cash-or-nothing call: n(d2) / (S·σ·√T).

    Interpretation: to delta-hedge N short contracts (each $1 face),
    hold N × binary_call_delta(S, K, σ, T) BTC units long.

    Returns 0.0 near expiry or with degenerate inputs.
    """
    if T < 1e-9 or sigma < 0.01 or S <= 0 or K <= 0:
        return 0.0
    d2 = (np.log(S / K) - 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    return float(norm.pdf(d2) / (S * sigma * np.sqrt(T)))


def binary_call_price(S: float, K: float, sigma: float, T: float) -> float:
    """
    Theoretical binary call price = N(d₂).
    Returns the risk-neutral probability of expiring ITM.
    Used during the intraday loop to check the early-exit condition.
    """
    if T < 1e-9 or sigma < 0.01 or S <= 0 or K <= 0:
        return 1.0 if S > K else 0.0
    d2 = (np.log(S / K) - 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    return float(norm.cdf(d2))


def kalshi_taker_fee(n_contracts: int, price_dollars: float) -> float:
    """
    Kalshi taker fee = 0.07 × C × P × (1 − P).
    Source: kalshi.com/fee-schedule (Feb 2026 schedule).
    Peaks at P = 0.50 → 1.75¢/contract; approaches zero near P = 0 or 1.
    price_dollars: contract price in dollars (e.g. 0.25 for 25-cent contract).
    """
    p = max(0.01, min(0.99, price_dollars))
    return KALSHI_TAKER_RATE * n_contracts * p * (1.0 - p)


def realized_vol_1h(entry_ts: datetime) -> float:
    """
    Annualised realised vol from the RV_LOOKBACK_MINS 1m Binance bars ending at entry_ts.
    Returns NaN if fewer than 10 bars are available (sparse data guard).
    """
    end_us   = int(_to_utc(entry_ts).timestamp() * 1_000_000)
    start_us = end_us - RV_LOOKBACK_MINS * 60 * 1_000_000
    bars = binance.filter(
        (pl.col("timestamp").cast(pl.Int64) >= start_us) &
        (pl.col("timestamp").cast(pl.Int64) <= end_us)
    ).sort("timestamp")
    if len(bars) < 10:
        return float("nan")
    closes      = bars["close"].to_numpy().astype(np.float64)
    log_returns = np.diff(np.log(closes))
    return float(np.std(log_returns, ddof=1) * np.sqrt(MINS_PER_YEAR))


def select_25d_contract(entry_bars: pl.DataFrame) -> pl.DataFrame | None:
    """
    From entry bars (multiple strikes, one expiry), return the single row
    representing the OTM +25Δ contract: strike > btc_close, IV ≥ IV_MIN,
    digi_px ≥ MIN_PREMIUM_CENTS, and digi_px closest to TARGET_DELTA_CENTS.
    Returns None if no qualifying OTM strike is found.
    """
    if len(entry_bars) == 0:
        return None
    btc = float(entry_bars["btc_close"][0])
    otm = entry_bars.filter(
        (pl.col("strike").cast(pl.Float64) > btc) &
        (pl.col("implied_vol").cast(pl.Float64) >= IV_MIN) &
        (pl.col("digi_px").cast(pl.Float64) >= MIN_PREMIUM_CENTS)
    )
    if len(otm) == 0:
        return None
    diffs = (otm["digi_px"].cast(pl.Float64) - TARGET_DELTA_CENTS).abs()
    return otm[int(diffs.arg_min())]


def get_btc_at(ts: datetime) -> float:
    """
    BTC close at the most recent completed 1m Binance bar at or before ts.
    Returns NaN if no bar found.
    """
    ts_us = int(_to_utc(ts).timestamp() * 1_000_000)
    sub = binance.filter(pl.col("timestamp").cast(pl.Int64) <= ts_us)
    if len(sub) == 0:
        return float("nan")
    return float(sub["close"][-1])


def get_iv_at(
    contract_surface: pl.DataFrame,
    target_ts: datetime,
    fallback_iv: float,
) -> float:
    """
    Most-recent implied_vol for a specific contract at or before target_ts.
    Falls back to fallback_iv if no prior bar is found or vol is invalid.
    """
    ts_us = int(_to_utc(target_ts).timestamp() * 1_000_000)
    sub = contract_surface.filter(
        pl.col("bar_ts").cast(pl.Int64) <= ts_us
    ).sort("bar_ts")
    if len(sub) == 0:
        return fallback_iv
    iv = float(sub["implied_vol"][-1])
    return iv if (not np.isnan(iv) and iv > 0.01) else fallback_iv


In [42]:
def run_single_contract(
    contract_surface: pl.DataFrame,
    entry_row: pl.DataFrame,
    expiry_ts: datetime,
) -> dict | None:
    """
    Simulate one hourly Kalshi binary-call contract:
      • Short N_CONTRACTS at entry (digi_px ≈ 25 cents, OTM)
      • Delta-hedge with BTC every HEDGE_INTERVAL_MINS minutes
      • Hold through expiration; apply binary settlement payout
      • Kalshi costs: $0.07/contract flat fee per side + 10 bps slippage on premium
      • BTC costs: 1 bps per trade (initial hedge, rebalances, unwind)

    Returns a trade record dict, or None if entry data is degenerate.
    """
    strike    = float(entry_row["strike"][0])
    entry_px  = float(entry_row["digi_px"][0])       # cents (0–100)
    entry_iv  = float(entry_row["implied_vol"][0])    # annualised decimal
    entry_btc = float(entry_row["btc_close"][0])
    entry_ts  = _to_utc(entry_row["bar_ts"][0])

    if any(np.isnan(v) for v in [entry_iv, entry_btc]) or entry_btc <= 0:
        return None

    expiry_ts = _to_utc(expiry_ts)

    # ── Entry ────────────────────────────────────────────────────────
    premium_received = N_CONTRACTS * entry_px / 100.0        # dollars
    kalshi_entry_fee = kalshi_taker_fee(N_CONTRACTS, entry_px / 100.0)
    premium_slippage = premium_received * CONTRACT_SLIPPAGE_BPS / 10_000
    contract_tc      = kalshi_entry_fee + premium_slippage

    T_entry   = max((expiry_ts - entry_ts).total_seconds() / (HOURS_IN_YEAR * 3600), 1e-9)
    delta_0   = binary_call_delta(entry_btc, strike, entry_iv, T_entry)
    btc_units = N_CONTRACTS * delta_0                         # BTC to hold long (hedge)
    btc_tc    = (btc_units * entry_btc) * BTC_SLIPPAGE_BPS / 10_000  # initial purchase TC

    # ── Intraday delta-hedging loop ──────────────────────────────────
    hedge_pnl    = 0.0
    prev_btc     = entry_btc
    exited_early = False

    rebalance_times: list[datetime] = []
    t = entry_ts + timedelta(minutes=HEDGE_INTERVAL_MINS)
    while t < expiry_ts:
        rebalance_times.append(t)
        t += timedelta(minutes=HEDGE_INTERVAL_MINS)

    for i, ts in enumerate(rebalance_times):
        curr_btc = get_btc_at(ts)
        if np.isnan(curr_btc):
            curr_btc = prev_btc

        # Mark-to-market BTC position
        hedge_pnl += btc_units * (curr_btc - prev_btc)

        T_curr  = max((expiry_ts - ts).total_seconds() / (HOURS_IN_YEAR * 3600), 1e-9)
        curr_iv = get_iv_at(contract_surface, ts, entry_iv)

        # ── Early exit: take profit at EARLY_EXIT_THRESHOLD of entry premium ──
        curr_contract_px = binary_call_price(curr_btc, strike, curr_iv, T_curr)
        if curr_contract_px <= (entry_px / 100.0) * EARLY_EXIT_THRESHOLD:
            buyback_cost  = N_CONTRACTS * curr_contract_px
            buyback_tc    = (kalshi_taker_fee(N_CONTRACTS, curr_contract_px)
                             + buyback_cost * CONTRACT_SLIPPAGE_BPS / 10_000)
            unwind_tc     = abs(btc_units * curr_btc) * BTC_SLIPPAGE_BPS / 10_000
            btc_tc       += unwind_tc
            total_tc      = contract_tc + btc_tc + buyback_tc
            trade_pnl     = (premium_received - contract_tc) + hedge_pnl - btc_tc - buyback_cost - buyback_tc
            exited_early  = True
            return {
                "expiry_ts":        expiry_ts,
                "expiry_hour_utc":  expiry_ts.hour,
                "trade_date":       expiry_ts.date(),
                "strike":           round(strike),
                "entry_px_cents":   round(entry_px, 2),
                "entry_iv":         round(entry_iv, 4),
                "entry_btc":        round(entry_btc, 1),
                "settlement_btc":   round(curr_btc, 1),
                "expired_itm":      False,
                "premium_received": round(premium_received, 4),
                "contract_tc":      round(contract_tc + buyback_tc, 6),
                "hedge_pnl":        round(hedge_pnl, 4),
                "btc_tc":           round(btc_tc, 4),
                "total_tc":         round(total_tc, 4),
                "settlement_pnl":   0.0,
                "pnl":              round(trade_pnl, 4),
                "n_hedges":         i + 1,
                "exited_early":     True,
            }

        # Normal rebalance
        new_btc_units = N_CONTRACTS * binary_call_delta(curr_btc, strike, curr_iv, T_curr)
        delta_usd     = abs(new_btc_units - btc_units) * curr_btc
        btc_tc       += delta_usd * BTC_SLIPPAGE_BPS / 10_000

        btc_units = new_btc_units
        prev_btc  = curr_btc

    # ── Settlement ───────────────────────────────────────────────────
    settlement_btc  = get_btc_at(expiry_ts)
    if np.isnan(settlement_btc):
        settlement_btc = prev_btc

    hedge_pnl += btc_units * (settlement_btc - prev_btc)
    unwind_tc  = abs(btc_units * settlement_btc) * BTC_SLIPPAGE_BPS / 10_000
    btc_tc    += unwind_tc

    expired_itm    = settlement_btc > strike
    settlement_pnl = -N_CONTRACTS * 1.0 if expired_itm else 0.0

    total_tc  = contract_tc + btc_tc
    trade_pnl = (premium_received - contract_tc) + hedge_pnl - btc_tc + settlement_pnl

    return {
        "expiry_ts":          expiry_ts,
        "expiry_hour_utc":    expiry_ts.hour,
        "trade_date":         expiry_ts.date(),
        "strike":             round(strike),
        "entry_px_cents":     round(entry_px, 2),
        "entry_iv":           round(entry_iv, 4),
        "entry_btc":          round(entry_btc, 1),
        "settlement_btc":     round(settlement_btc, 1),
        "expired_itm":        expired_itm,
        "premium_received":   round(premium_received, 4),
        "contract_tc":        round(contract_tc, 6),
        "hedge_pnl":          round(hedge_pnl, 4),
        "btc_tc":             round(btc_tc, 4),
        "total_tc":           round(total_tc, 4),
        "settlement_pnl":     round(settlement_pnl, 2),
        "pnl":                round(trade_pnl, 4),
        "n_hedges":           len(rebalance_times),
        "exited_early":       False,
    }


In [43]:
def run_full_backtest(
    surface: pl.DataFrame,
    session_hours: list[int] | None = None,
) -> pl.DataFrame:
    """
    Iterate over all unique expiry times in vol_surface, select the +25Δ OTM
    contract, run the delta-hedged backtest for each.

    Args:
        session_hours: if set, only trade contracts whose expiry hour (UTC)
                       is in this list.  None = all 24 hours.

    Returns:
        pl.DataFrame with one row per trade.
    """
    trades: list[dict] = []
    expiry_times = surface["expiry_time"].unique().sort().to_list()
    label = "all hours" if session_hours is None else f"hours {session_hours[0]}–{session_hours[-1]} UTC"
    print(f"Running backtest ({label}) over {len(expiry_times)} unique expiries...")

    for expiry_ts in expiry_times:
        expiry_ts = _to_utc(expiry_ts)

        if session_hours is not None and expiry_ts.hour not in session_hours:
            continue

        grp = surface.filter(pl.col("expiry_time") == expiry_ts)
        max_mte    = int(grp["minutes_to_expiry"].max())
        entry_bars = grp.filter(pl.col("minutes_to_expiry") == max_mte)

        entry_row = select_25d_contract(entry_bars)
        if entry_row is None:
            continue

        # ── IV / Realised-vol filter ──────────────────────────────────
        entry_ts_val = _to_utc(entry_row["bar_ts"][0])
        rv = realized_vol_1h(entry_ts_val)
        if not np.isnan(rv) and rv > 0:
            if float(entry_row["implied_vol"][0]) / rv < IV_RVOL_RATIO_MIN:
                continue   # implied vol not rich enough — skip

        contract_name    = str(entry_row["digi_contract_name"][0])
        contract_surface = grp.filter(pl.col("digi_contract_name") == contract_name)

        record = run_single_contract(contract_surface, entry_row, expiry_ts)
        if record is not None:
            trades.append(record)

    if not trades:
        print("No trades generated — check data / filter.")
        return pl.DataFrame()

    df = pl.DataFrame(trades).sort("expiry_ts")
    print(f"Generated {len(df):,} trades.")
    return df


In [44]:
trades = run_full_backtest(surface, session_hours=None)
trades.head(10)


Running backtest (all hours) over 1380 unique expiries...
Generated 78 trades.


expiry_ts,expiry_hour_utc,trade_date,strike,entry_px_cents,entry_iv,entry_btc,settlement_btc,expired_itm,premium_received,contract_tc,hedge_pnl,btc_tc,total_tc,settlement_pnl,pnl,n_hedges,exited_early
"datetime[μs, UTC]",i64,date,i64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,f64,i64,bool
2026-03-22 14:00:00 UTC,14,2026-03-22,69100,10.0,0.4866,68646.0,68783.4,false,100.0,9.16662,32.6294,15.6235,24.7901,0.0,67.2484,7,true
2026-03-22 21:00:00 UTC,21,2026-03-22,68350,36.0,0.4646,68230.4,68190.8,false,360.0,17.793514,-58.6974,72.1487,89.9422,0.0,192.6269,11,true
2026-03-24 09:00:00 UTC,9,2026-03-24,71300,42.0,0.6991,71196.3,71132.0,false,420.0,27.987754,-29.7478,16.0979,44.0856,0.0,166.0846,3,true
2026-03-27 21:00:00 UTC,21,2026-03-27,66400,10.0,0.4117,66030.6,65920.2,false,100.0,8.469922,-75.0281,11.3199,19.7898,0.0,-24.8616,5,true
2026-03-27 22:00:00 UTC,22,2026-03-27,66600,29.0,1.4307,66051.4,66087.0,false,290.0,16.036005,12.1594,4.518,20.554,0.0,262.4696,1,true
2026-03-31 02:00:00 UTC,2,2026-03-31,67700,26.0,0.9877,67249.6,67111.6,false,260.0,15.21468,-63.5963,6.1937,21.4084,0.0,153.6051,1,true
2026-03-31 04:00:00 UTC,4,2026-03-31,68400,24.0,0.9326,67927.8,67857.3,false,240.0,14.402318,-32.6589,6.2902,20.6925,0.0,166.6145,1,true
2026-03-31 14:00:00 UTC,14,2026-03-31,67200,11.0,0.5541,66719.1,67522.8,true,110.0,6.963,671.6149,38.674,45.637,-1000.0,-264.0221,11,false
2026-04-02 00:00:00 UTC,0,2026-04-02,69700,37.0,7.1885,68158.3,68101.4,false,370.0,24.145478,-6.6487,1.9778,26.1233,0.0,218.2142,11,true


In [45]:
def compute_metrics(trades_df: pl.DataFrame) -> dict:
    """Compute aggregate performance metrics for a set of trades."""
    if len(trades_df) == 0:
        return {}

    pnl = trades_df["pnl"].cast(pl.Float64).to_numpy()
    cum = np.cumsum(pnl)

    daily = (
        trades_df
        .with_columns(pl.col("trade_date").cast(pl.Utf8))
        .group_by("trade_date")
        .agg(pl.col("pnl").sum().alias("daily_pnl"))
        .sort("trade_date")
    )
    daily_pnl = daily["daily_pnl"].cast(pl.Float64).to_numpy()
    n_days    = len(daily_pnl)

    total_ret   = float(cum[-1])
    ann_pct     = (total_ret / STARTING_PORTFOLIO) * (252 / n_days) * 100 if n_days > 0 else 0.0
    std_daily   = float(np.std(daily_pnl, ddof=1))
    sharpe      = float(np.mean(daily_pnl) / std_daily * np.sqrt(252)) if std_daily > 1e-9 else float("nan")
    win_rate    = float(np.mean(pnl > 0)) * 100
    running_max = np.maximum.accumulate(cum)
    max_dd      = float((cum - running_max).min())
    calmar      = total_ret / abs(max_dd) if max_dd < 0 else float("nan")
    itm_rate       = float(trades_df["expired_itm"].mean()) * 100
    early_exit_pct = float(trades_df["exited_early"].mean()) * 100

    return {
        "n_trades":            len(trades_df),
        "total_return_$":      round(total_ret, 2),
        "annualised_pct":      round(ann_pct, 2),
        "sharpe_ratio":        round(sharpe, 3) if not np.isnan(sharpe) else "n/a",
        "win_rate_pct":        round(win_rate, 1),
        "max_drawdown_$":      round(max_dd, 2),
        "calmar_ratio":        round(calmar, 3) if not np.isnan(calmar) else "n/a",
        "itm_rate_pct":        round(itm_rate, 1),
        "early_exit_pct":      round(early_exit_pct, 1),
        "avg_pnl_$":           round(float(np.mean(pnl)), 4),
        "avg_hedge_pnl_$":     round(float(trades_df["hedge_pnl"].cast(pl.Float64).mean()), 4),
        "avg_premium_recv_$":  round(float(trades_df["premium_received"].cast(pl.Float64).mean()), 4),
        "total_tc_$":          round(float(trades_df["total_tc"].cast(pl.Float64).sum()), 2),
    }


In [46]:
metrics = compute_metrics(trades)
print("\n=== BACKTEST RESULTS — SELL +25Δ VOL, ALL HOURS ===")
print(f"Portfolio start: ${STARTING_PORTFOLIO:,.0f}  |  N contracts/trade: {N_CONTRACTS:,}")
print("-" * 55)
for k, v in metrics.items():
    print(f"  {k:<30} {v}")



=== BACKTEST RESULTS — SELL +25Δ VOL, ALL HOURS ===
Portfolio start: $100,000  |  N contracts/trade: 1,000
-------------------------------------------------------
  n_trades                       78
  total_return_$                 4246.49
  annualised_pct                 27.44
  sharpe_ratio                   4.759
  win_rate_pct                   78.2
  max_drawdown_$                 -3283.79
  calmar_ratio                   1.293
  itm_rate_pct                   11.5
  early_exit_pct                 83.3
  avg_pnl_$                      54.4422
  avg_hedge_pnl_$                33.491
  avg_premium_recv_$             238.9744
  total_tc_$                     3708.83


## Cross-Validation by UTC Expiry Hour

In [47]:
def cv_by_hour(trades_df: pl.DataFrame) -> pd.DataFrame:
    """Per-UTC-hour metrics for cross-validation."""
    rows = []
    for hour in range(24):
        sub = trades_df.filter(pl.col("expiry_hour_utc") == hour)
        if len(sub) == 0:
            continue
        m = compute_metrics(sub)
        m["hour_utc"] = hour
        rows.append(m)
    return pd.DataFrame(rows).set_index("hour_utc")

hour_cv = cv_by_hour(trades)
display(hour_cv[["n_trades", "sharpe_ratio", "win_rate_pct", "itm_rate_pct", "total_return_$"]])

fig_hour = px.bar(
    hour_cv.reset_index(), x="hour_utc", y="sharpe_ratio",
    color="sharpe_ratio", color_continuous_scale="RdYlGn",
    title="Sharpe Ratio by Expiry Hour (UTC) — Cross-Validation",
    labels={"hour_utc": "Expiry Hour (UTC)", "sharpe_ratio": "Sharpe Ratio"},
)
fig_hour.add_hline(y=0, line_dash="dash", line_color="black")
fig_hour.show()


,n_trades,sharpe_ratio,win_rate_pct,itm_rate_pct,total_return_$
hour_utc,,,,,
0,2,-6.6490,50.0000,50.0000,-634.2400
1,3,1.2430,66.7000,33.3000,23.8500
2,6,25.1270,100.0000,0.0000,1053.2500
3,6,-0.4490,83.3000,16.7000,-62.8000
4,4,30.4030,100.0000,0.0000,645.0200
8,2,20.5340,100.0000,0.0000,157.8500
9,4,34.5860,100.0000,0.0000,640.1800
10,3,11.8520,66.7000,33.3000,374.2000
11,2,19.5450,100.0000,0.0000,230.4200


## Bootstrap Cross-Validation

Randomly select 50% of trades (without replacement) and re-compute Sharpe. Repeat 1 000 times. If the strategy is driven by a handful of lucky hours, the distribution will be wide and centred near 0.

In [48]:
rng     = np.random.default_rng(42)
N_BOOT  = 1_000
FRAC    = 0.50   # 50% random subsample per iteration

sharpe_boot: list[float] = []
for _ in range(N_BOOT):
    n_sample = max(2, int(len(trades) * FRAC))
    idx      = rng.choice(len(trades), size=n_sample, replace=False)
    sample   = trades[idx.tolist()]
    m        = compute_metrics(sample)
    s        = m.get("sharpe_ratio", "n/a")
    if isinstance(s, (int, float)) and not np.isnan(float(s)):
        sharpe_boot.append(float(s))

sharpe_boot_arr = np.array(sharpe_boot)
p5, p50, p95    = np.percentile(sharpe_boot_arr, [5, 50, 95])

print(f"Bootstrap Sharpe Distribution ({N_BOOT} iterations, {int(FRAC*100)}% random subsample):")
print(f"  P5:       {p5:>8.3f}")
print(f"  Median:   {p50:>8.3f}")
print(f"  P95:      {p95:>8.3f}")
print(f"  Pct > 0:  {(sharpe_boot_arr > 0).mean() * 100:>6.1f}%")

fig_boot = px.histogram(
    x=sharpe_boot_arr, nbins=60,
    title=f"Bootstrap Sharpe Distribution (n={N_BOOT}, {int(FRAC*100)}% samples each)",
    labels={"x": "Sharpe Ratio"},
)
fig_boot.add_vline(x=0,   line_dash="dash", line_color="red",  annotation_text="0")
fig_boot.add_vline(x=p50, line_dash="dot",  line_color="blue", annotation_text=f"Median {p50:.2f}")
fig_boot.show()


Bootstrap Sharpe Distribution (1000 iterations, 50% random subsample):
  P5:          0.624
  Median:      4.233
  P95:        10.870
  Pct > 0:    97.6%


## Visualisations

In [49]:
trades_pd = trades.to_pandas()
trades_pd["expiry_ts"] = pd.to_datetime(trades_pd["expiry_ts"], utc=True)
trades_pd = trades_pd.sort_values("expiry_ts").reset_index(drop=True)

# Cumulative P&L decomposition
trades_pd["cum_total"]      = trades_pd["pnl"].cumsum()
trades_pd["cum_premium"]    = (trades_pd["premium_received"] - trades_pd["contract_tc"]).cumsum()
trades_pd["cum_hedge"]      = trades_pd["hedge_pnl"].cumsum()
trades_pd["cum_settlement"] = trades_pd["settlement_pnl"].cumsum()
trades_pd["cum_tc"]         = (-trades_pd["total_tc"]).cumsum()

fig_cum = go.Figure()
for col, name, dash in [
    ("cum_total",      "Total P&L",      "solid"),
    ("cum_premium",    "Premium (net)",  "dot"),
    ("cum_hedge",      "Hedge P&L",      "dash"),
    ("cum_settlement", "Settlement P&L", "dashdot"),
    ("cum_tc",         "Cum. TC (neg.)", "longdash"),
]:
    fig_cum.add_trace(go.Scatter(
        x=trades_pd["expiry_ts"], y=trades_pd[col],
        name=name, line=dict(dash=dash, width=2),
    ))
fig_cum.update_layout(
    title="Cumulative P&L Decomposition",
    xaxis_title="Date", yaxis_title="$ P&L",
    hovermode="x unified",
)
fig_cum.show()

# Drawdown
cum_arr     = trades_pd["cum_total"].values
running_max = np.maximum.accumulate(cum_arr)
drawdown    = cum_arr - running_max
fig_dd = px.area(
    x=trades_pd["expiry_ts"], y=drawdown,
    title="Drawdown ($)", labels={"x": "Date", "y": "Drawdown ($)"},
    color_discrete_sequence=["crimson"],
)
fig_dd.show()

# Trade P&L histogram
fig_hist = px.histogram(
    trades_pd, x="pnl", nbins=60,
    title="Trade P&L Distribution ($)",
    labels={"pnl": "Trade P&L ($)"},
)
fig_hist.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Break-even")
fig_hist.show()

# Win rate heatmap: expiry hour × month
trades_pd["month"] = trades_pd["expiry_ts"].dt.strftime("%Y-%m")
pivot = (
    trades_pd
    .groupby(["expiry_hour_utc", "month"])["pnl"]
    .apply(lambda x: (x > 0).mean())
    .unstack(fill_value=np.nan)
)
fig_heat = px.imshow(
    pivot,
    title="Win Rate by Expiry Hour (UTC) × Month",
    labels=dict(x="Month", y="Expiry Hour (UTC)", color="Win Rate"),
    color_continuous_scale="RdYlGn", zmin=0, zmax=1,
    text_auto=".0%",
)
fig_heat.show()
